In [ ]:
# Dependencies are managed by requirements-mac.txt; no installs during execution.


In [ ]:
from imblearn.over_sampling import SMOTE

## Step 1: Data Loading and Initial Setup

In [ ]:
# Run from the repository root or Training_Report directory.
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "game_theory").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "game_theory").is_dir():
    raise RuntimeError("Open notebook from the project root or Training_Report")
sys.path.insert(0, str(PROJECT_ROOT))
from tools.data_pipeline import load_splits, preprocess_splits, prepare_features, hybrid_balance
from game_theory.ids_training import train_ids, metrics, predict_labels, save_json
from game_theory.training_bridge import StackelbergIDSController


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("Loading pre-split CICIoT2023 datasets...")

# QARMOR_DATA_DIR or original Kaggle paths; no replacement data.
train_df, val_df, test_df = load_splits()

print(f"Training set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")
print(f"Test set shape: {test_df.shape}")

# Display basic info
print(f"\nTotal samples: {train_df.shape[0] + val_df.shape[0] + test_df.shape[0]:,}")
print(f"Total features: {train_df.shape[1] - 1}")  # Excluding label column


## Step 2: Data Exploration and Quality Assessment

In [ ]:
def explore_dataset(df, dataset_name):
    """Explore dataset characteristics"""
    print(f"\n=== {dataset_name} Dataset Analysis ===")
    print(f"Shape: {df.shape}")
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # Check for missing values
    missing_values = df.isnull().sum()
    if missing_values.sum() > 0:
        print(f"\nMissing values found: {missing_values.sum()}")
        print(missing_values[missing_values > 0])
    else:
        print("No missing values found")
    
    # Check for duplicates
    duplicates = df.duplicated().sum()
    print(f"Duplicate rows: {duplicates}")
    
    # Check data types
    print(f"\nData types:")
    print(df.dtypes.value_counts())
    
    return df

# Explore all datasets
train_df = explore_dataset(train_df, "Training")
val_df = explore_dataset(val_df, "Validation") 
test_df = explore_dataset(test_df, "Test")

# Analyze target variable distribution
print("\n=== Target Variable Analysis ===")
print("Training set label distribution:")
print(train_df['label'].value_counts())

print("\nValidation set label distribution:")
print(val_df['label'].value_counts())

print("\nTest set label distribution:")
print(test_df['label'].value_counts())

# Get unique attack types across all datasets
all_labels = pd.concat([train_df['label'], val_df['label'], test_df['label']]).unique()
print(f"\nTotal unique attack types: {len(all_labels)}")
print("Attack types:", sorted(all_labels))


## Step 3: Data Preprocessing Pipeline

In [ ]:
combined_df, label_encoder = preprocess_splits(train_df, val_df, test_df)
class_labels = label_encoder.classes_.tolist()


## Step 4: Feature Engineering and Selection

In [ ]:
X_processed, selected_features, scaler, selector = prepare_features(combined_df, n_features=25, seed=42)
feature_importance = list(zip(selected_features, selector.scores_[selector.get_support()]))


In [ ]:
# Heatmap of selected features
# Subset the combined_df to selected features only (excluding label and others)
feature_data = combined_df[selected_features]

plt.figure(figsize=(14, 12))
corr_matrix = feature_data.corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', square=True, cbar=True)
plt.title('Correlation Heatmap of Selected 25 Features')
plt.tight_layout()
plt.show()

In [ ]:
# Removed fabricated importance scores and selected-feature overwrite; measured scores are in feature_importance.


In [ ]:
features, scores = zip(*sorted(feature_importance, key=lambda x: x[1], reverse=True))
plt.figure(figsize=(14,6))
plt.barh(features, scores)
plt.title("Training-only mutual information")
plt.show()


In [ ]:
# Pairplot or Distribution Plot of Top Features
# For visualization performance, sample 50,000 records randomly
sample_df = combined_df[combined_df.dataset.eq("train")].sample(n=min(5000, sum(combined_df.dataset.eq("train"))), random_state=42)

# Use top 5 features for pairplot (too many features will overwhelm)
top_5_features = [f[0] for f in feature_importance[:5]]
top_5_features.append('label')

import seaborn as sns
sns.pairplot(sample_df[top_5_features], hue='label', corner=True, plot_kws={'alpha':0.3, 's':10})
plt.suptitle('Pairplot of Top 5 Features Colored by Class', y=1.02)
plt.show()

## Step 5: Split Back to Original Datasets

In [ ]:
def split_processed_data(X_processed):
    """
    Split processed data back to train, validation, and test sets
    """
    print("Splitting processed data back to original sets...")
    
    # Split back to original datasets
    train_processed = X_processed[X_processed['dataset'] == 'train'].drop('dataset', axis=1)
    val_processed = X_processed[X_processed['dataset'] == 'val'].drop('dataset', axis=1)
    test_processed = X_processed[X_processed['dataset'] == 'test'].drop('dataset', axis=1)
    
    # Separate features and labels
    feature_cols = [col for col in train_processed.columns if col != 'label_encoded']
    
    X_train = train_processed[feature_cols].values
    y_train = train_processed['label_encoded'].values
    
    X_val = val_processed[feature_cols].values
    y_val = val_processed['label_encoded'].values
    
    X_test = test_processed[feature_cols].values
    y_test = test_processed['label_encoded'].values
    
    print(f"Training set: {X_train.shape}")
    print(f"Validation set: {X_val.shape}")  
    print(f"Test set: {X_test.shape}")
    
    # Convert to categorical for multi-class classification
    num_classes = len(label_encoder.classes_)
    y_train_cat = to_categorical(y_train, num_classes=num_classes)
    y_val_cat = to_categorical(y_val, num_classes=num_classes)
    y_test_cat = to_categorical(y_test, num_classes=num_classes)
    
    print(f"Number of classes: {num_classes}")
    
    return (X_train, y_train, y_train_cat, 
            X_val, y_val, y_val_cat,
            X_test, y_test, y_test_cat, 
            num_classes)

# Split the processed data
(X_train, y_train, y_train_cat, 
 X_val, y_val, y_val_cat,
 X_test, y_test, y_test_cat, 
 num_classes) = split_processed_data(X_processed)


## Step 6: Build CNN-LSTM-Attention Model

In [ ]:
from tools.ids_model import build_enhanced_cnn_lstm_attention_model
num_classes = len(class_labels)
model = build_enhanced_cnn_lstm_attention_model((X_train.shape[1],), num_classes)
model.summary()


In [ ]:
# Class Distribution
import matplotlib.pyplot as plt
import seaborn as sns

# Combine all labels from the dataset
labels = combined_df['label']

plt.figure(figsize=(16, 6))
sns.countplot(data=combined_df, y='label', order=labels.value_counts().index, palette='viridis')
plt.title('Class Distribution in the Dataset')
plt.xlabel('Number of Samples')
plt.ylabel('Attack Type')
plt.tight_layout()
plt.show()

## Hybrid SMOTE

In [ ]:
# Balancing runs once in the following cell. Original variants retained in backups.


In [ ]:
X_train_balanced, y_train_balanced = hybrid_balance(X_train, y_train, seed=42)
y_train_balanced_int = y_train_balanced


## Model Training

In [ ]:
from tensorflow.keras.utils import to_categorical

# One-hot encode labels
y_train_balanced_cat = to_categorical(y_train_balanced, num_classes=num_classes)
y_val_cat = to_categorical(y_val, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)
# ----------------------------------------------------------
# Ensure you have integer label versions (missing earlier)
# ----------------------------------------------------------
y_train_balanced_int = y_train_balanced
y_val_int = y_val
y_test_int = y_test

In [ ]:
# Standard baseline and three-seed comparison: python -m tools.run_ablation
# Do not pretrain the proposed model before its feedback loop.


## Co-Evolutionary Game Theory Integrated DL Framework

In [ ]:
# Maintained implementation: actual validation predictions -> game -> next epoch.
from tensorflow.keras.optimizers import Adam
model.compile(optimizer=Adam(learning_rate=5e-4, clipnorm=1.0),
              loss="categorical_crossentropy", metrics=["accuracy"])
history_log, controller = train_ids(
    model, X_train_balanced, y_train_balanced_int, X_val, y_val,
    label_encoder.classes_.tolist(), epochs=10, batch_fraction=0.10,
    seed=42, output_dir=PROJECT_ROOT / "results/notebook",
    controller_options={"scale_step": 0.15, "max_scale": 2.5,
                        "available_defenses": ["NORMAL", "CLASS_REWEIGHTING"]})
# Evaluate test once after validation-selected checkpoint restoration.
y_true = y_test
y_pred = predict_labels(model, X_test, class_labels)
test_results = metrics(y_true, y_pred, class_labels)
cm = np.asarray(test_results["confusion_matrix"])
report_dict = test_results["per_class"]
save_json(PROJECT_ROOT / "results/notebook/test.json", test_results)


Historical extension (inactive; see backups).

```python
# ==================================================================
# STEP 8 → STEP 11 (Co-evolutionary training + entropy anneal +
# mini-batch + per-class payoff plots + strategy/utilities tracking)
# ==================================================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.optimizers import Adam
import time, os

# -----------------------
# Robust LR helpers (unchanged)
# -----------------------
def get_current_lr(optimizer):
    try:
        return float(tf.keras.backend.get_value(optimizer.learning_rate))
    except Exception:
        try:
            return float(optimizer.learning_rate)
        except Exception:
            return None

def set_current_lr(optimizer, new_lr):
    try:
        tf.keras.backend.set_value(optimizer.learning_rate, new_lr)
        return True
    except Exception:
        try:
            optimizer.learning_rate = new_lr
            return True
        except Exception:
            return False

# -----------------------
# Setup training config (unchanged)
# -----------------------
def setup_enhanced_training_configuration():
    class_weights = compute_class_weight(
        'balanced',
        classes=np.unique(y_train_balanced),
        y=y_train_balanced
    )
    class_weight_dict = {cls: float(weight) for cls, weight in zip(np.unique(y_train_balanced), class_weights)}

    model.compile(
        optimizer=Adam(learning_rate=0.0005, clipnorm=1.0),
        loss='categorical_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
            MacroF1(num_classes=num_classes)
        ]
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_macro_f1', patience=5, restore_best_weights=True, mode='max'),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_macro_f1', factor=0.3, patience=8, min_lr=1e-8, mode='max'),
        tf.keras.callbacks.ModelCheckpoint('best_enhanced_iot_ids_model.h5', monitor='val_macro_f1', save_best_only=True, mode='max')
    ]

    training_params = {
        'batch_size': 256,
        'epochs': 1,  # coevolution will control outer loop
        'callbacks': callbacks,
        'validation_data': (X_val, y_val_cat),
        'verbose': 1,
        'shuffle': True,
        'class_weight': class_weight_dict
    }
    return training_params

# -----------------------
# Macro-F1 (unchanged)
# -----------------------
class MacroF1(tf.keras.metrics.Metric):
    def __init__(self, num_classes, name="macro_f1", **kwargs):
        super(MacroF1, self).__init__(name=name, **kwargs)
        self.num_classes = int(num_classes)
        self.tp = self.add_weight(name='tp', shape=(self.num_classes,), initializer='zeros')
        self.fp = self.add_weight(name='fp', shape=(self.num_classes,), initializer='zeros')
        self.fn = self.add_weight(name='fn', shape=(self.num_classes,), initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true_idx = tf.argmax(y_true, axis=-1)
        y_pred_idx = tf.argmax(y_pred, axis=-1)
        y_true_oh = tf.one_hot(y_true_idx, depth=self.num_classes, dtype=tf.float32)
        y_pred_oh = tf.one_hot(y_pred_idx, depth=self.num_classes, dtype=tf.float32)

        tp_batch = tf.reduce_sum(y_true_oh * y_pred_oh, axis=0)
        fp_batch = tf.reduce_sum((1.0 - y_true_oh) * y_pred_oh, axis=0)
        fn_batch = tf.reduce_sum(y_true_oh * (1.0 - y_pred_oh), axis=0)

        self.tp.assign_add(tp_batch)
        self.fp.assign_add(fp_batch)
        self.fn.assign_add(fn_batch)

    def result(self):
        precision = tf.math.divide_no_nan(self.tp, self.tp + self.fp)
        recall = tf.math.divide_no_nan(self.tp, self.tp + self.fn)
        f1 = tf.math.divide_no_nan(2.0 * precision * recall, precision + recall)
        return tf.reduce_mean(f1)

    def reset_state(self):
        self.tp.assign(tf.zeros_like(self.tp))
        self.fp.assign(tf.zeros_like(self.fp))
        self.fn.assign(tf.zeros_like(self.fn))

    def reset_states(self):
        self.reset_state()

# -----------------------
# Replicator update with entropy annealing + clipping
# -----------------------
def replicator_update(probs, utilities, entropy_reg=0.02, min_prob=1e-3, eps=1e-9):
    probs = np.array(probs, dtype=float)
    utilities = np.array(utilities, dtype=float)

    # ensure non-negative utilities
    min_u = utilities.min()
    if min_u <= 0:
        utilities = utilities - min_u + 1e-8

    avg_u = np.dot(probs, utilities) + eps
    new = probs * (utilities / avg_u)
    new = new / (new.sum() + eps)

    # entropy reg (mix with uniform)
    if entropy_reg is not None and entropy_reg > 0:
        uniform = np.ones_like(new) / len(new)
        new = (1.0 - entropy_reg) * new + entropy_reg * uniform

    # clipping to ensure minimal exploration
    new = np.clip(new, min_prob, 1.0)
    new = new / (new.sum() + eps)
    return new

# -----------------------
# helper to compute DR/FAR/MR (unchanged)
# -----------------------
def compute_per_class_rates(y_true, y_pred, num_classes):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    total = cm.sum()
    DR = np.zeros(num_classes, dtype=float)
    FAR = np.zeros(num_classes, dtype=float)
    MR = np.zeros(num_classes, dtype=float)
    for c in range(num_classes):
        TP = cm[c, c]
        FN = cm[c, :].sum() - TP
        FP = cm[:, c].sum() - TP
        TN = total - TP - FP - FN
        DR[c] = TP / (TP + FN + 1e-12)
        FAR[c] = FP / (FP + TN + 1e-12)
        MR[c] = 1.0 - DR[c]
    return DR, FAR, MR, cm

# -----------------------
# default strategy matrices (unchanged)
# -----------------------
def default_strategy_matrices(num_def_strategies, num_att_strategies, num_classes, class_counts=None):
    if class_counts is None:
        class_counts = np.ones(num_classes)
    def_w = np.ones((num_def_strategies, num_classes), dtype=float)
    minority_importance = (np.max(class_counts) / (class_counts + 1e-8))
    minority_importance = minority_importance / (minority_importance.mean() + 1e-12)
    if num_def_strategies > 1:
        def_w[1, :] = 1.0 + 1.5 * (minority_importance - 1.0)
    if num_def_strategies > 2:
        def_w[2, :] = 1.0 + 0.7 * (1.0 / (1.0 + minority_importance))
    att_w = np.ones((num_att_strategies, num_classes), dtype=float)
    if num_att_strategies > 1:
        att_w[1, :] = 1.0 + 1.8 * (minority_importance - 1.0)
    def_w = def_w / def_w.sum(axis=1, keepdims=True)
    att_w = att_w / att_w.sum(axis=1, keepdims=True)
    return def_w, att_w

# -----------------------
# Per-class payoff plotting helper (saves file)
# -----------------------
def save_per_class_payoff_plot(y_true, y_pred, epoch, out_dir="coevo_reports", class_names=None):
    os.makedirs(out_dir, exist_ok=True)
    prec, rec, f1, sup = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)
    if class_names is None:
        class_names = [str(i) for i in range(len(f1))]

    plt.figure(figsize=(10,4))
    plt.bar(class_names, f1)
    plt.ylim(0,1)
    plt.xlabel("Class")
    plt.ylabel("F1 (Defender payoff)")
    plt.title(f"Per-Class Payoff (F1) at Epoch {epoch}")
    plt.xticks(rotation=60)
    out_path = os.path.join(out_dir, f"payoff_epoch_{epoch}.png")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()
    return out_path

# -----------------------
# Summary validation report (printing + returns dict)
# -----------------------
def print_validation_report(y_true, y_pred, prefix="Validation", class_names=None):
    acc = accuracy_score(y_true, y_pred)
    prec_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    print(f"\n--- {prefix} Summary ---")
    print(f"Accuracy: {acc:.4f} | Precision (macro): {prec_macro:.4f} | Recall (macro): {rec_macro:.4f} | Macro-F1: {f1_macro:.4f}")
    print(f"Payoff -> Defender: {f1_macro:.4f} | Attacker: {1.0 - f1_macro:.4f}")
    print("\nClassification Report:\n")
    print(classification_report(y_true, y_pred, target_names=class_names if class_names else None, digits=4, zero_division=0))
    return {"accuracy": acc, "precision_macro": prec_macro, "recall_macro": rec_macro, "f1_macro": f1_macro}

# -----------------------
# Main co-evolutionary training loop (updated)
# -----------------------
def train_coevolutionary(
    model,
    X_train, y_train_int, y_train_cat,
    X_val, y_val_int, y_val_cat,
    num_classes,
    training_params,
    epochs_coev=50,
    num_def_strats=3,
    num_att_strats=2,
    verbose_epoch_log=True,
    per10_plot=True,
    corrective_actions=True,
    patience_bad=3,
    drop_lr_factor=0.5,
    min_lr=1e-8,
    scale_init=1.0,
    scale_increase=1.25,
    save_best_path='coev_best_weights.weights.h5',
    batch_fraction=0.1,       # fraction of training data used per coevo epoch
    entropy_init=0.05,        # starting entropy regularization
    entropy_min=0.005,        # minimum entropy when annealed
    entropy_decay=0.97,       # multiplicative decay per epoch
    min_prob=1e-3,
    report_every=10,
    class_names=None
):
    np.random.seed(42)

    # strategy distributions initialization
    p_def = np.ones(num_def_strats) / num_def_strats
    p_att = np.ones(num_att_strats) / num_att_strats

    class_counts = np.bincount(y_train_int, minlength=num_classes)
    def_strategy_weights, att_strategy_weights = default_strategy_matrices(
        num_def_strats, num_att_strats, num_classes, class_counts
    )

    history_log = {
        "epoch": [], "p_def": [], "p_att": [], "macro_f1_val": [],
        "per_class_f1": [], "val_confusion": [], "U_def": [], "U_att": [],
        "val_acc": [], "val_prec_macro": [], "val_rec_macro": [], "defender_payoff": [], "attacker_payoff": []
    }

    best_val = -1.0
    bad_count = 0
    adaptive_scale = scale_init

    n_train = len(X_train)

    for epoch in range(epochs_coev):
        start = time.time()

        # anneal entropy reg
        entropy_reg = max(entropy_min, entropy_init * (entropy_decay ** epoch))

        # defender class multiplier mapping
        class_multiplier = 1.0 + adaptive_scale * np.dot(p_def, def_strategy_weights)
        class_multiplier = class_multiplier / (np.mean(class_multiplier) + 1e-12)
        class_weight_dict = {i: float(class_multiplier[i]) for i in range(num_classes)}

        # mini-batch sample (or full-batch if fraction>=1)
        if batch_fraction is None or batch_fraction >= 1.0:
            X_batch = X_train
            y_batch = y_train_cat
        else:
            batch_size_total = max(1, int(n_train * batch_fraction))
            idx = np.random.choice(n_train, size=batch_size_total, replace=False)
            X_batch = X_train[idx]
            y_batch = y_train_cat[idx]

        # train 1 epoch on sampled batch
        model.fit(
            X_batch, y_batch,
            batch_size=training_params.get('batch_size', 256),
            epochs=1,
            verbose=0,
            shuffle=training_params.get('shuffle', True),
            class_weight=class_weight_dict,
            callbacks=[]
        )

        # validation predictions
        y_val_pred_probs = model.predict(X_val, batch_size=1024, verbose=0)
        y_val_pred = np.argmax(y_val_pred_probs, axis=1)

        # per-class rates
        DR, FAR, MR, cm_val = compute_per_class_rates(y_val_int, y_val_pred, num_classes)

        # utilities per class
        alpha, beta, gamma = 0.8, 0.4, 0.6
        u_def_class = alpha * DR - beta * FAR
        u_att_class = (1.0 - DR) + gamma * MR

        # map class utilities -> strategy utilities
        U_def_strat = np.dot(def_strategy_weights, u_def_class)
        U_att_strat = np.dot(att_strategy_weights, u_att_class)

        # replicator update with annealed entropy/clipping
        p_def = replicator_update(p_def, U_def_strat, entropy_reg=entropy_reg, min_prob=min_prob)
        p_att = replicator_update(p_att, U_att_strat, entropy_reg=entropy_reg, min_prob=min_prob)

        # logging metrics
        val_macro_f1 = f1_score(y_val_int, y_val_pred, average='macro', zero_division=0)
        per_class_f1 = f1_score(y_val_int, y_val_pred, average=None, labels=list(range(num_classes)), zero_division=0)

        acc = accuracy_score(y_val_int, y_val_pred)
        prec_macro = precision_score(y_val_int, y_val_pred, average='macro', zero_division=0)
        rec_macro = recall_score(y_val_int, y_val_pred, average='macro', zero_division=0)
        defender_payoff = val_macro_f1
        attacker_payoff = 1.0 - defender_payoff

        history_log["epoch"].append(epoch + 1)
        history_log["p_def"].append(p_def.copy())
        history_log["p_att"].append(p_att.copy())
        history_log["macro_f1_val"].append(val_macro_f1)
        history_log["per_class_f1"].append(per_class_f1.copy())
        history_log["val_confusion"].append(cm_val.copy())
        history_log["U_def"].append(U_def_strat.copy())
        history_log["U_att"].append(U_att_strat.copy())
        history_log["val_acc"].append(acc)
        history_log["val_prec_macro"].append(prec_macro)
        history_log["val_rec_macro"].append(rec_macro)
        history_log["defender_payoff"].append(defender_payoff)
        history_log["attacker_payoff"].append(attacker_payoff)

        # checkpoint & corrective actions
        improved = val_macro_f1 > (best_val + 1e-8)
        if improved:
            best_val = val_macro_f1
            bad_count = 0
            try:
                model.save_weights(save_best_path)
            except Exception:
                pass
        else:
            bad_count += 1

        if corrective_actions and (bad_count >= patience_bad):
            old_lr = get_current_lr(model.optimizer)
            if old_lr is None:
                new_lr = None
            else:
                new_lr = max(old_lr * drop_lr_factor, min_lr)
                set_current_lr(model.optimizer, new_lr)
            adaptive_scale *= scale_increase
            bad_count = 0
            if verbose_epoch_log:
                print(f"--> Corrective action at epoch {epoch+1}: lr {old_lr} -> {new_lr}, scale->{adaptive_scale:.3f}")

        # verbose
        if verbose_epoch_log:
            print(f"[CoEvo Epoch {epoch+1:03d}] time={time.time()-start:.2f}s val_macro_f1={val_macro_f1:.4f} val_acc={acc:.4f} "
                  f"entropy={entropy_reg:.4f} p_def={np.round(p_def,3)} p_att={np.round(p_att,3)}")

        # periodic extended report & per-class payoff visualization
        if report_every and ((epoch + 1) % report_every == 0):
            save_path = save_per_class_payoff_plot(y_val_int, y_val_pred, epoch + 1, out_dir="coevo_reports", class_names=class_names)
            print_validation_report(y_val_int, y_val_pred, prefix=f"Epoch {epoch+1} Validation", class_names=class_names)
            print(f"Saved per-class payoff plot to: {save_path}")

        # per-10-epoch visual (keeps original layout)
        if per10_plot and ((epoch + 1) % 10 == 0 or epoch == epochs_coev - 1):
            fig, axes = plt.subplots(3, 2, figsize=(16, 14))
            axes[0,0].plot(history_log["epoch"], history_log["macro_f1_val"], marker='o'); axes[0,0].set_title("Validation Macro-F1 (co-evo)")
            pdef_arr = np.vstack(history_log["p_def"]); patt_arr = np.vstack(history_log["p_att"])
            for i in range(pdef_arr.shape[1]): axes[0,1].plot(history_log["epoch"], pdef_arr[:, i], label=f'p_def_{i}')
            for j in range(patt_arr.shape[1]): axes[0,1].plot(history_log["epoch"], patt_arr[:, j], '--', label=f'p_att_{j}')
            axes[0,1].legend(fontsize='small'); axes[0,1].set_title("Strategy Distributions")
            Udef_arr = np.vstack(history_log["U_def"]); Uatt_arr = np.vstack(history_log["U_att"])
            for i in range(Udef_arr.shape[1]): axes[1,0].plot(history_log["epoch"], Udef_arr[:, i], label=f'U_def_{i}')
            for j in range(Uatt_arr.shape[1]): axes[1,1].plot(history_log["epoch"], Uatt_arr[:, j], '--', label=f'U_att_{j}')
            axes[1,0].legend(fontsize='small'); axes[1,1].legend(fontsize='small')
            axes[1,0].set_title("Defender Utilities"); axes[1,1].set_title("Attacker Utilities")
            sns.heatmap(history_log["val_confusion"][-1], ax=axes[2,0], cmap="Blues"); axes[2,0].set_title("Confusion Matrix (last)")
            last_f1 = history_log["per_class_f1"][-1]; axes[2,1].bar(range(num_classes), last_f1); axes[2,1].set_ylim(0,1); axes[2,1].set_title("Per-Class F1 (last)")
            plt.tight_layout(); plt.show()

    # restore best
    if os.path.exists(save_best_path):
        try:
            model.load_weights(save_best_path)
        except Exception:
            pass

    return model, history_log

# -----------------------
# Wrapper + final eval (unchanged except params forwarded)
# -----------------------
def train_coevolutionary_with_visualization_wrapper(
    model,
    X_train_balanced, y_train_balanced_int, y_train_balanced_cat,
    X_val, y_val_int, y_val_cat,
    X_test, y_test_int, y_test_cat,
    num_classes,
    epochs_coev=50,
    batch_fraction=0.1,
    entropy_init=0.05,
    entropy_min=0.005,
    entropy_decay=0.97,
    min_prob=1e-3,
    report_every=10,
    class_names=None
):
    training_params = setup_enhanced_training_configuration()
    model, history_log = train_coevolutionary(
        model,
        X_train_balanced, y_train_balanced_int, y_train_balanced_cat,
        X_val, y_val_int, y_val_cat,
        num_classes=num_classes,
        training_params=training_params,
        epochs_coev=epochs_coev,
        num_def_strats=3,
        num_att_strats=2,
        verbose_epoch_log=True,
        per10_plot=True,
        corrective_actions=True,
        patience_bad=3,
        drop_lr_factor=0.5,
        min_lr=1e-8,
        scale_init=1.0,
        scale_increase=1.25,
        save_best_path = "coev_best_weights.weights.h5",
        batch_fraction=batch_fraction,
        entropy_init=entropy_init,
        entropy_min=entropy_min,
        entropy_decay=entropy_decay,
        min_prob=min_prob,
        report_every=report_every,
        class_names=class_names
    )
    # final eval on test
    report_dict, cm = evaluate_model_full_quick(model, X_test, y_test_cat, class_names=class_names)
    return model, history_log, report_dict, cm

def evaluate_model_full_quick(model, X_test, y_test_cat, class_names=None):
    y_pred_probs = model.predict(X_test, batch_size=1024, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = np.argmax(y_test_cat, axis=1)
    report_dict = classification_report(
        y_true, y_pred,
        target_names=class_names if class_names else None,
        digits=4, output_dict=True, zero_division=0
    )
    print("\n📊 Per-Class Performance Report (Test):\n")
    print(classification_report(
        y_true, y_pred,
        target_names=class_names if class_names else None,
        digits=4, zero_division=0
    ))
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 8)); sns.heatmap(cm, annot=False, cmap="Blues", fmt='d'); plt.title("Confusion Matrix on Test"); plt.xlabel("Predicted"); plt.ylabel("True"); plt.show()
    f1_scores = [report_dict[str(i)]['f1-score'] for i in range(len(report_dict) - 3)]
    plt.figure(figsize=(14, 6)); sns.barplot(x=list(range(len(f1_scores))), y=f1_scores); plt.title("Per-Class F1 on Test"); plt.xlabel("Class"); plt.ylabel("F1 Score"); plt.ylim(0, 1); plt.xticks(rotation=90); plt.show()
    return report_dict, cm

# ==========================================================
# TRAINING INITIATION (Step 8 → Step 11)
# ==========================================================
training_params = setup_enhanced_training_configuration()
# Recompile safe
model.compile(optimizer=Adam(learning_rate=5e-4, clipnorm=1.0),
              loss='categorical_crossentropy',
              metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
                       tf.keras.metrics.Recall(name='recall'), MacroF1(num_classes=num_classes)])

# Run training
model, history_log, report_dict, cm = train_coevolutionary_with_visualization_wrapper(
    model,
    X_train_balanced, y_train_balanced, y_train_balanced_cat,
    X_val, y_val, y_val_cat,
    X_test, y_test, y_test_cat,
    num_classes,
    epochs_coev=50,      # outer co-evolution rounds
    batch_fraction=0.1,  # 10% per coevo epoch
    entropy_init=0.05,
    entropy_min=0.005,
    entropy_decay=0.97,
    min_prob=1e-3,
    report_every=10,
    class_names=None
)
# Save final model
model.save("final_coevo_iot_ids_model.h5")

```


Historical extension (inactive; see backups).

```python
# ==================================================================
# STEP 8 → STEP 11 (Co-evolutionary training + entropy anneal +
# mini-batch + per-class payoff plots + strategy/utilities tracking)
# ==================================================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.optimizers import Adam
import time, os

# -----------------------
# Robust LR helpers (unchanged)
# -----------------------
def get_current_lr(optimizer):
    try:
        return float(tf.keras.backend.get_value(optimizer.learning_rate))
    except Exception:
        try:
            return float(optimizer.learning_rate)
        except Exception:
            return None

def set_current_lr(optimizer, new_lr):
    try:
        tf.keras.backend.set_value(optimizer.learning_rate, new_lr)
        return True
    except Exception:
        try:
            optimizer.learning_rate = new_lr
            return True
        except Exception:
            return False

# -----------------------
# Setup training config (unchanged)
# -----------------------
def setup_enhanced_training_configuration():
    class_weights = compute_class_weight(
        'balanced',
        classes=np.unique(y_train_balanced),
        y=y_train_balanced
    )
    class_weight_dict = {cls: float(weight) for cls, weight in zip(np.unique(y_train_balanced), class_weights)}

    model.compile(
        optimizer=Adam(learning_rate=0.0005, clipnorm=1.0),
        loss='categorical_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
            MacroF1(num_classes=num_classes)
        ]
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_macro_f1', patience=5, restore_best_weights=True, mode='max'),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_macro_f1', factor=0.3, patience=8, min_lr=1e-8, mode='max'),
        tf.keras.callbacks.ModelCheckpoint('best_enhanced_iot_ids_model.h5', monitor='val_macro_f1', save_best_only=True, mode='max')
    ]

    training_params = {
        'batch_size': 256,
        'epochs': 1,  # coevolution will control outer loop
        'callbacks': callbacks,
        'validation_data': (X_val, y_val_cat),
        'verbose': 1,
        'shuffle': True,
        'class_weight': class_weight_dict
    }
    return training_params

# -----------------------
# Macro-F1 (unchanged)
# -----------------------
class MacroF1(tf.keras.metrics.Metric):
    def __init__(self, num_classes, name="macro_f1", **kwargs):
        super(MacroF1, self).__init__(name=name, **kwargs)
        self.num_classes = int(num_classes)
        self.tp = self.add_weight(name='tp', shape=(self.num_classes,), initializer='zeros')
        self.fp = self.add_weight(name='fp', shape=(self.num_classes,), initializer='zeros')
        self.fn = self.add_weight(name='fn', shape=(self.num_classes,), initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true_idx = tf.argmax(y_true, axis=-1)
        y_pred_idx = tf.argmax(y_pred, axis=-1)
        y_true_oh = tf.one_hot(y_true_idx, depth=self.num_classes, dtype=tf.float32)
        y_pred_oh = tf.one_hot(y_pred_idx, depth=self.num_classes, dtype=tf.float32)

        tp_batch = tf.reduce_sum(y_true_oh * y_pred_oh, axis=0)
        fp_batch = tf.reduce_sum((1.0 - y_true_oh) * y_pred_oh, axis=0)
        fn_batch = tf.reduce_sum(y_true_oh * (1.0 - y_pred_oh), axis=0)

        self.tp.assign_add(tp_batch)
        self.fp.assign_add(fp_batch)
        self.fn.assign_add(fn_batch)

    def result(self):
        precision = tf.math.divide_no_nan(self.tp, self.tp + self.fp)
        recall = tf.math.divide_no_nan(self.tp, self.tp + self.fn)
        f1 = tf.math.divide_no_nan(2.0 * precision * recall, precision + recall)
        return tf.reduce_mean(f1)

    def reset_state(self):
        self.tp.assign(tf.zeros_like(self.tp))
        self.fp.assign(tf.zeros_like(self.fp))
        self.fn.assign(tf.zeros_like(self.fn))

    def reset_states(self):
        self.reset_state()

# -----------------------
# Replicator update with entropy annealing + clipping
# -----------------------
def replicator_update(probs, utilities, entropy_reg=0.02, min_prob=1e-3, eps=1e-9):
    probs = np.array(probs, dtype=float)
    utilities = np.array(utilities, dtype=float)

    # ensure non-negative utilities
    min_u = utilities.min()
    if min_u <= 0:
        utilities = utilities - min_u + 1e-8

    avg_u = np.dot(probs, utilities) + eps
    new = probs * (utilities / avg_u)
    new = new / (new.sum() + eps)

    # entropy reg (mix with uniform)
    if entropy_reg is not None and entropy_reg > 0:
        uniform = np.ones_like(new) / len(new)
        new = (1.0 - entropy_reg) * new + entropy_reg * uniform

    # clipping to ensure minimal exploration
    new = np.clip(new, min_prob, 1.0)
    new = new / (new.sum() + eps)
    return new

# -----------------------
# helper to compute DR/FAR/MR (unchanged)
# -----------------------
def compute_per_class_rates(y_true, y_pred, num_classes):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    total = cm.sum()
    DR = np.zeros(num_classes, dtype=float)
    FAR = np.zeros(num_classes, dtype=float)
    MR = np.zeros(num_classes, dtype=float)
    for c in range(num_classes):
        TP = cm[c, c]
        FN = cm[c, :].sum() - TP
        FP = cm[:, c].sum() - TP
        TN = total - TP - FP - FN
        DR[c] = TP / (TP + FN + 1e-12)
        FAR[c] = FP / (FP + TN + 1e-12)
        MR[c] = 1.0 - DR[c]
    return DR, FAR, MR, cm

# -----------------------
# default strategy matrices (unchanged)
# -----------------------
def default_strategy_matrices(num_def_strategies, num_att_strategies, num_classes, class_counts=None):
    if class_counts is None:
        class_counts = np.ones(num_classes)
    def_w = np.ones((num_def_strategies, num_classes), dtype=float)
    minority_importance = (np.max(class_counts) / (class_counts + 1e-8))
    minority_importance = minority_importance / (minority_importance.mean() + 1e-12)
    if num_def_strategies > 1:
        def_w[1, :] = 1.0 + 1.5 * (minority_importance - 1.0)
    if num_def_strategies > 2:
        def_w[2, :] = 1.0 + 0.7 * (1.0 / (1.0 + minority_importance))
    att_w = np.ones((num_att_strategies, num_classes), dtype=float)
    if num_att_strategies > 1:
        att_w[1, :] = 1.0 + 1.8 * (minority_importance - 1.0)
    def_w = def_w / def_w.sum(axis=1, keepdims=True)
    att_w = att_w / att_w.sum(axis=1, keepdims=True)
    return def_w, att_w

# -----------------------
# Per-class payoff plotting helper (saves file)
# -----------------------
def save_per_class_payoff_plot(y_true, y_pred, epoch, out_dir="coevo_reports", class_names=None):
    os.makedirs(out_dir, exist_ok=True)
    prec, rec, f1, sup = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)
    if class_names is None:
        class_names = [str(i) for i in range(len(f1))]

    plt.figure(figsize=(10,4))
    plt.bar(class_names, f1)
    plt.ylim(0,1)
    plt.xlabel("Class")
    plt.ylabel("F1 (Defender payoff)")
    plt.title(f"Per-Class Payoff (F1) at Epoch {epoch}")
    plt.xticks(rotation=60)
    out_path = os.path.join(out_dir, f"payoff_epoch_{epoch}.png")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()
    return out_path

# -----------------------
# Summary validation report (printing + returns dict)
# -----------------------
def print_validation_report(y_true, y_pred, prefix="Validation", class_names=None):
    acc = accuracy_score(y_true, y_pred)
    prec_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    print(f"\n--- {prefix} Summary ---")
    print(f"Accuracy: {acc:.4f} | Precision (macro): {prec_macro:.4f} | Recall (macro): {rec_macro:.4f} | Macro-F1: {f1_macro:.4f}")
    print(f"Payoff -> Defender: {f1_macro:.4f} | Attacker: {1.0 - f1_macro:.4f}")
    print("\nClassification Report:\n")
    print(classification_report(y_true, y_pred, target_names=class_names if class_names else None, digits=4, zero_division=0))
    return {"accuracy": acc, "precision_macro": prec_macro, "recall_macro": rec_macro, "f1_macro": f1_macro}

# -----------------------
# Main co-evolutionary training loop (updated)
# -----------------------
def train_coevolutionary(
    model,
    X_train, y_train_int, y_train_cat,
    X_val, y_val_int, y_val_cat,
    num_classes,
    training_params,
    epochs_coev=50,
    num_def_strats=3,
    num_att_strats=2,
    verbose_epoch_log=True,
    per10_plot=True,
    corrective_actions=True,
    patience_bad=3,
    drop_lr_factor=0.5,
    min_lr=1e-8,
    scale_init=1.0,
    scale_increase=1.25,
    save_best_path='coev_best_weights.weights.h5',
    batch_fraction=0.1,       # fraction of training data used per coevo epoch
    entropy_init=0.05,        # starting entropy regularization
    entropy_min=0.005,        # minimum entropy when annealed
    entropy_decay=0.97,       # multiplicative decay per epoch
    min_prob=1e-3,
    report_every=10,
    class_names=None
):
    np.random.seed(42)

    # strategy distributions initialization
    p_def = np.ones(num_def_strats) / num_def_strats
    p_att = np.ones(num_att_strats) / num_att_strats

    class_counts = np.bincount(y_train_int, minlength=num_classes)
    def_strategy_weights, att_strategy_weights = default_strategy_matrices(
        num_def_strats, num_att_strats, num_classes, class_counts
    )

    history_log = {
        "epoch": [], "p_def": [], "p_att": [], "macro_f1_val": [],
        "per_class_f1": [], "val_confusion": [], "U_def": [], "U_att": [],
        "val_acc": [], "val_prec_macro": [], "val_rec_macro": [], "defender_payoff": [], "attacker_payoff": []
    }

    best_val = -1.0
    bad_count = 0
    adaptive_scale = scale_init

    n_train = len(X_train)

    for epoch in range(epochs_coev):
        start = time.time()

        # anneal entropy reg
        entropy_reg = max(entropy_min, entropy_init * (entropy_decay ** epoch))

        # defender class multiplier mapping
        class_multiplier = 1.0 + adaptive_scale * np.dot(p_def, def_strategy_weights)
        class_multiplier = class_multiplier / (np.mean(class_multiplier) + 1e-12)
        class_weight_dict = {i: float(class_multiplier[i]) for i in range(num_classes)}

        # mini-batch sample (or full-batch if fraction>=1)
        if batch_fraction is None or batch_fraction >= 1.0:
            X_batch = X_train
            y_batch = y_train_cat
        else:
            batch_size_total = max(1, int(n_train * batch_fraction))
            idx = np.random.choice(n_train, size=batch_size_total, replace=False)
            X_batch = X_train[idx]
            y_batch = y_train_cat[idx]

        # train 1 epoch on sampled batch
        model.fit(
            X_batch, y_batch,
            batch_size=training_params.get('batch_size', 256),
            epochs=1,
            verbose=0,
            shuffle=training_params.get('shuffle', True),
            class_weight=class_weight_dict,
            callbacks=[]
        )

        # validation predictions
        y_val_pred_probs = model.predict(X_val, batch_size=1024, verbose=0)
        y_val_pred = np.argmax(y_val_pred_probs, axis=1)

        # per-class rates
        DR, FAR, MR, cm_val = compute_per_class_rates(y_val_int, y_val_pred, num_classes)

        # utilities per class
        alpha, beta, gamma = 0.8, 0.4, 0.6
        u_def_class = alpha * DR - beta * FAR
        u_att_class = (1.0 - DR) + gamma * MR

        # map class utilities -> strategy utilities
        U_def_strat = np.dot(def_strategy_weights, u_def_class)
        U_att_strat = np.dot(att_strategy_weights, u_att_class)

        # replicator update with annealed entropy/clipping
        p_def = replicator_update(p_def, U_def_strat, entropy_reg=entropy_reg, min_prob=min_prob)
        p_att = replicator_update(p_att, U_att_strat, entropy_reg=entropy_reg, min_prob=min_prob)

        # logging metrics
        val_macro_f1 = f1_score(y_val_int, y_val_pred, average='macro', zero_division=0)
        per_class_f1 = f1_score(y_val_int, y_val_pred, average=None, labels=list(range(num_classes)), zero_division=0)

        acc = accuracy_score(y_val_int, y_val_pred)
        prec_macro = precision_score(y_val_int, y_val_pred, average='macro', zero_division=0)
        rec_macro = recall_score(y_val_int, y_val_pred, average='macro', zero_division=0)
        defender_payoff = val_macro_f1
        attacker_payoff = 1.0 - defender_payoff

        history_log["epoch"].append(epoch + 1)
        history_log["p_def"].append(p_def.copy())
        history_log["p_att"].append(p_att.copy())
        history_log["macro_f1_val"].append(val_macro_f1)
        history_log["per_class_f1"].append(per_class_f1.copy())
        history_log["val_confusion"].append(cm_val.copy())
        history_log["U_def"].append(U_def_strat.copy())
        history_log["U_att"].append(U_att_strat.copy())
        history_log["val_acc"].append(acc)
        history_log["val_prec_macro"].append(prec_macro)
        history_log["val_rec_macro"].append(rec_macro)
        history_log["defender_payoff"].append(defender_payoff)
        history_log["attacker_payoff"].append(attacker_payoff)

        # checkpoint & corrective actions
        improved = val_macro_f1 > (best_val + 1e-8)
        if improved:
            best_val = val_macro_f1
            bad_count = 0
            try:
                model.save_weights(save_best_path)
            except Exception:
                pass
        else:
            bad_count += 1

        if corrective_actions and (bad_count >= patience_bad):
            old_lr = get_current_lr(model.optimizer)
            if old_lr is None:
                new_lr = None
            else:
                new_lr = max(old_lr * drop_lr_factor, min_lr)
                set_current_lr(model.optimizer, new_lr)
            adaptive_scale *= scale_increase
            bad_count = 0
            if verbose_epoch_log:
                print(f"--> Corrective action at epoch {epoch+1}: lr {old_lr} -> {new_lr}, scale->{adaptive_scale:.3f}")

        # verbose
        if verbose_epoch_log:
            print(f"[CoEvo Epoch {epoch+1:03d}] time={time.time()-start:.2f}s val_macro_f1={val_macro_f1:.4f} val_acc={acc:.4f} "
                  f"entropy={entropy_reg:.4f} p_def={np.round(p_def,3)} p_att={np.round(p_att,3)}")

        # periodic extended report & per-class payoff visualization
        if report_every and ((epoch + 1) % report_every == 0):
            save_path = save_per_class_payoff_plot(y_val_int, y_val_pred, epoch + 1, out_dir="coevo_reports", class_names=class_names)
            print_validation_report(y_val_int, y_val_pred, prefix=f"Epoch {epoch+1} Validation", class_names=class_names)
            print(f"Saved per-class payoff plot to: {save_path}")

        # per-10-epoch visual (keeps original layout)
        if per10_plot and ((epoch + 1) % 10 == 0 or epoch == epochs_coev - 1):
            fig, axes = plt.subplots(3, 2, figsize=(16, 14))
            axes[0,0].plot(history_log["epoch"], history_log["macro_f1_val"], marker='o'); axes[0,0].set_title("Validation Macro-F1 (co-evo)")
            pdef_arr = np.vstack(history_log["p_def"]); patt_arr = np.vstack(history_log["p_att"])
            for i in range(pdef_arr.shape[1]): axes[0,1].plot(history_log["epoch"], pdef_arr[:, i], label=f'p_def_{i}')
            for j in range(patt_arr.shape[1]): axes[0,1].plot(history_log["epoch"], patt_arr[:, j], '--', label=f'p_att_{j}')
            axes[0,1].legend(fontsize='small'); axes[0,1].set_title("Strategy Distributions")
            Udef_arr = np.vstack(history_log["U_def"]); Uatt_arr = np.vstack(history_log["U_att"])
            for i in range(Udef_arr.shape[1]): axes[1,0].plot(history_log["epoch"], Udef_arr[:, i], label=f'U_def_{i}')
            for j in range(Uatt_arr.shape[1]): axes[1,1].plot(history_log["epoch"], Uatt_arr[:, j], '--', label=f'U_att_{j}')
            axes[1,0].legend(fontsize='small'); axes[1,1].legend(fontsize='small')
            axes[1,0].set_title("Defender Utilities"); axes[1,1].set_title("Attacker Utilities")
            sns.heatmap(history_log["val_confusion"][-1], ax=axes[2,0], cmap="Blues"); axes[2,0].set_title("Confusion Matrix (last)")
            last_f1 = history_log["per_class_f1"][-1]; axes[2,1].bar(range(num_classes), last_f1); axes[2,1].set_ylim(0,1); axes[2,1].set_title("Per-Class F1 (last)")
            plt.tight_layout(); plt.show()

    # restore best
    if os.path.exists(save_best_path):
        try:
            model.load_weights(save_best_path)
        except Exception:
            pass

    return model, history_log

# -----------------------
# Wrapper + final eval (unchanged except params forwarded)
# -----------------------
def train_coevolutionary_with_visualization_wrapper(
    model,
    X_train_balanced, y_train_balanced_int, y_train_balanced_cat,
    X_val, y_val_int, y_val_cat,
    X_test, y_test_int, y_test_cat,
    num_classes,
    epochs_coev=50,
    batch_fraction=0.1,
    entropy_init=0.05,
    entropy_min=0.005,
    entropy_decay=0.97,
    min_prob=1e-3,
    report_every=10,
    class_names=None
):
    training_params = setup_enhanced_training_configuration()
    model, history_log = train_coevolutionary(
        model,
        X_train_balanced, y_train_balanced_int, y_train_balanced_cat,
        X_val, y_val_int, y_val_cat,
        num_classes=num_classes,
        training_params=training_params,
        epochs_coev=epochs_coev,
        num_def_strats=3,
        num_att_strats=2,
        verbose_epoch_log=True,
        per10_plot=True,
        corrective_actions=True,
        patience_bad=3,
        drop_lr_factor=0.5,
        min_lr=1e-8,
        scale_init=1.0,
        scale_increase=1.25,
        save_best_path = "coev_best_weights.weights.h5",
        batch_fraction=batch_fraction,
        entropy_init=entropy_init,
        entropy_min=entropy_min,
        entropy_decay=entropy_decay,
        min_prob=min_prob,
        report_every=report_every,
        class_names=class_names
    )
    # final eval on test
    report_dict, cm = evaluate_model_full_quick(model, X_test, y_test_cat, class_names=class_names)
    return model, history_log, report_dict, cm

def evaluate_model_full_quick(model, X_test, y_test_cat, class_names=None):
    y_pred_probs = model.predict(X_test, batch_size=1024, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = np.argmax(y_test_cat, axis=1)
    report_dict = classification_report(
        y_true, y_pred,
        target_names=class_names if class_names else None,
        digits=4, output_dict=True, zero_division=0
    )
    print("\n📊 Per-Class Performance Report (Test):\n")
    print(classification_report(
        y_true, y_pred,
        target_names=class_names if class_names else None,
        digits=4, zero_division=0
    ))
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 8)); sns.heatmap(cm, annot=False, cmap="Blues", fmt='d'); plt.title("Confusion Matrix on Test"); plt.xlabel("Predicted"); plt.ylabel("True"); plt.show()
    f1_scores = [report_dict[str(i)]['f1-score'] for i in range(len(report_dict) - 3)]
    plt.figure(figsize=(14, 6)); sns.barplot(x=list(range(len(f1_scores))), y=f1_scores); plt.title("Per-Class F1 on Test"); plt.xlabel("Class"); plt.ylabel("F1 Score"); plt.ylim(0, 1); plt.xticks(rotation=90); plt.show()
    return report_dict, cm

# ==========================================================
# TRAINING INITIATION (Step 8 → Step 11)
# ==========================================================
training_params = setup_enhanced_training_configuration()
# Recompile safe
model.compile(optimizer=Adam(learning_rate=5e-4, clipnorm=1.0),
              loss='categorical_crossentropy',
              metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
                       tf.keras.metrics.Recall(name='recall'), MacroF1(num_classes=num_classes)])

# Run training
model, history_log, report_dict, cm = train_coevolutionary_with_visualization_wrapper(
    model,
    X_train_balanced, y_train_balanced, y_train_balanced_cat,
    X_val, y_val, y_val_cat,
    X_test, y_test, y_test_cat,
    num_classes,
    epochs_coev=50,      # outer co-evolution rounds
    batch_fraction=0.1,  # 10% per coevo epoch
    entropy_init=0.05,
    entropy_min=0.005,
    entropy_decay=0.97,
    min_prob=1e-3,
    report_every=10,
    class_names=None
)
# Save final model
model.save("final_coevo_iot_ids_model.h5")

```


In [ ]:
import joblib
joblib.dump(scaler, str(PROJECT_ROOT / "results/notebook/scaler.pkl"))


In [ ]:
joblib.dump(label_encoder, str(PROJECT_ROOT / "results/notebook/label_encoder.pkl"))

In [ ]:
import json
json.dump(selected_features, open(str(PROJECT_ROOT / "results/notebook/features.json"),"w"))

In [ ]:
joblib.dump(history_log, str(PROJECT_ROOT / "results/notebook/training_history.pkl"))

In [ ]:
np.save(str(PROJECT_ROOT / "results/notebook/confusion_matrix.npy"), cm)

In [ ]:
import json
json.dump(report_dict, open(str(PROJECT_ROOT / "results/notebook/classification_report.json"),"w"))

Historical extension (inactive; see backups).

```python
# ===========================================================
# 🧠 REBUILD Co-Evolutionary CNN–BiLSTM IDS FOR FEDERATED LEARNING
# ===========================================================
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Dropout, Bidirectional, LSTM, Dense, Flatten, concatenate
import numpy as np
import os

# ✅ Use absolute Kaggle paths
model_path = "/kaggle/working/final_coevo_iot_ids_model.h5"
weights_path = "/kaggle/working/coev_best_weights.weights.h5"

# ===========================================================
# 1️⃣ Try to directly load your saved .h5 model (simplest path)
# ===========================================================
try:
    global_model = tf.keras.models.load_model(model_path, compile=False)
    print("✅ Loaded full model from .h5 successfully!")
except Exception as e:
    print(f"⚠️ Could not load full model. Falling back to manual rebuild.\nError: {e}")
    
    # Manual rebuild (approximate architecture)
    inp = Input(shape=(100, 1))
    x = Conv1D(128, 3, activation='relu', padding='same')(inp)
    x = Conv1D(256, 3, activation='relu', padding='same')(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)
    x = Bidirectional(LSTM(256, return_sequences=True))(x)
    x = Bidirectional(LSTM(256))(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.4)(x)
    out = Dense(5, activation='softmax')(x)
    global_model = Model(inputs=inp, outputs=out)
    print("✅ Manually rebuilt Co-Evo CNN–BiLSTM model.")

# ===========================================================
# 2️⃣ Load weights safely
# ===========================================================
try:
    global_model.load_weights(weights_path)
    print("✅ Loaded weights successfully from:", weights_path)
except Exception as e:
    print(f"⚠️ Could not load weights exactly. Error:\n{e}")

# ===========================================================
# 3️⃣ Federated Learning Simulation
# ===========================================================
def simulate_client_data(num_clients=5, samples_per_client=400):
    X_clients = [np.random.rand(samples_per_client, 100, 1) for _ in range(num_clients)]
    y_clients = [np.random.randint(0, 5, size=(samples_per_client,)) for _ in range(num_clients)]
    return X_clients, y_clients

X_clients, y_clients = simulate_client_data()

def federated_avg(global_model, X_clients, y_clients, epochs=1):
    global_weights = global_model.get_weights()
    new_weights = []

    for i, (Xc, yc) in enumerate(zip(X_clients, y_clients)):
        local_model = tf.keras.models.clone_model(global_model)
        local_model.set_weights(global_weights)
        local_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
        local_model.fit(Xc, yc, epochs=epochs, batch_size=32, verbose=0)
        new_weights.append(local_model.get_weights())
        print(f"✅ Client-{i+1} trained locally and sent weights.")

    averaged = [np.mean(np.array(weights), axis=0) for weights in zip(*new_weights)]
    global_model.set_weights(averaged)
    print("🌍 Global model updated through Federated Averaging.")
    return global_model

# 🔄 Run 1 federated learning round (demo)
global_model = federated_avg(global_model, X_clients, y_clients, epochs=1)

```


Historical extension (inactive; see backups).

```python
#===========================================
```


Historical extension (inactive; see backups).

```python
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model

def coevo_attention_block():
    """Implements co-evolutionary attention mechanism."""
    def attention_func(inputs):
        q, k, v = inputs
        score = tf.matmul(q, k, transpose_b=True)
        score = score / tf.sqrt(tf.cast(tf.shape(k)[-1], tf.float32))
        weights = tf.nn.softmax(score, axis=-1)
        context = tf.matmul(weights, v)
        return context
    return Lambda(attention_func, name="coevo_attention")

def rebuild_coevo_cnn_bilstm(input_shape=(100, 1), num_classes=5):
    inputs = Input(shape=input_shape, name="input_layer")
    x = Reshape((input_shape[0], input_shape[1]))(inputs)

    # ---- CNN feature extractor ----
    for filters in [64, 128, 256]:
        y = Conv1D(filters, 3, activation='relu', padding='same')(x)
        y = BatchNormalization()(y)
        y = Conv1D(filters, 3, activation='relu', padding='same')(y)
        x = Add()([x, y]) if x.shape[-1] == filters else y
        x = MaxPooling1D(pool_size=2)(x)
        x = Dropout(0.25)(x)

    # ---- BiLSTM sequence learner ----
    x = LSTM(128, return_sequences=True)(x)
    x = LayerNormalization()(x)
    x = Bidirectional(LSTM(64, return_sequences=True))(x)
    x = LayerNormalization()(x)
    x = LSTM(32, return_sequences=True)(x)

    # ---- Attention (Co-Evolutionary Adaptation) ----
    query = Dense(64)(x)
    key = Dense(64)(x)
    value = Dense(64)(x)

    context = coevo_attention_block()([query, key, value])

    # 🔧 Project both to same dimension before Add()
    x_proj = Dense(64)(x)
    x = Add()([x_proj, context])
    x = LayerNormalization()(x)
    x = GlobalAveragePooling1D()(x)

    # ---- Dense Layers ----
    x = Dense(128, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    outputs = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs, outputs, name="Rebuilt_CoEvo_CNN_BiLSTM")
    return model


# ✅ Rebuild and load weights safely
global_model = rebuild_coevo_cnn_bilstm(input_shape=(100, 1), num_classes=5)

# Load your pretrained weights
global_model.load_weights("/kaggle/working/coev_best_weights.weights.h5", skip_mismatch=True)

print("✅ Model rebuilt & weights loaded successfully!")
global_model.summary()

```


Historical extension (inactive; see backups).

```python
# =====================================================
# FEDERATED LEARNING INTEGRATION (TensorFlow Federated)
# =====================================================

!pip install tensorflow-federated --quiet

import tensorflow as tf
import tensorflow_federated as tff

# ----- Reuse your existing model -----
def create_keras_model():
    return rebuild_coevo_cnn_bilstm(input_shape=(100, 1), num_classes=5)

# ----- Wrap model for TFF -----
def model_fn():
    keras_model = create_keras_model()
    return tff.learning.models.from_keras_model(
        keras_model,
        input_spec=sample_batch,  # We'll define a sample batch below
        loss=tf.keras.losses.CategoricalCrossentropy(),
        metrics=[tf.keras.metrics.CategoricalAccuracy()]
    )

# ---- Create Mock Client Datasets (simulation only) ----
def create_mock_client_data(num_clients=5, num_samples=200):
    import numpy as np
    clients = []
    for i in range(num_clients):
        X = np.random.rand(num_samples, 100, 1).astype('float32')
        y = tf.keras.utils.to_categorical(np.random.randint(0, 5, num_samples), 5)
        ds = tf.data.Dataset.from_tensor_slices((X, y))
        ds = ds.shuffle(100).batch(16)
        clients.append(ds)
    return clients

client_datasets = create_mock_client_data()

# Extract one batch to define input_spec
sample_batch = client_datasets[0].element_spec

# ---- Create Federated Learning Process ----
iterative_process = tff.learning.algorithms.build_weighted_fed_avg(
    model_fn=model_fn,
    client_optimizer_fn=lambda: tf.keras.optimizers.Adam(learning_rate=1e-3),
    server_optimizer_fn=lambda: tf.keras.optimizers.SGD(learning_rate=1.0)
)

# ---- Initialize & Run Federated Rounds ----
state = iterative_process.initialize()

NUM_ROUNDS = 5
for round_num in range(1, NUM_ROUNDS + 1):
    state, metrics = iterative_process.next(state, client_datasets)
    print(f"Round {round_num} - Metrics: {metrics}")

```


Historical extension (inactive; see backups).

```python
import h5py
import json
import os
from tensorflow.keras.models import load_model

def inspect_h5_model(file_path):
    print(f"\n--- Inspecting HDF5 file: {file_path} ---")

    if not os.path.exists(file_path):
        print("❌ File not found.")
        return

    with h5py.File(file_path, 'r') as f:
        print("\nTop-level keys:", list(f.keys()))
        print("\nTop-level attributes:")
        for k, v in f.attrs.items():
            print(f"  {k} : {v if k != 'model_config' else '<<model_config JSON omitted for brevity>>'}")

        # Optional: Save model_config to JSON for inspection
        if 'model_config' in f.attrs:
            model_config = json.loads(f.attrs['model_config'])
            model_name = model_config['config'].get('name', 'Unknown_Model')
            print(f"\nModel Name: {model_name}")
            print("\nLayer Summary:")
            for layer in model_config['config']['layers']:
                layer_name = layer.get('config', {}).get('name', 'Unnamed')
                layer_type = layer.get('class_name', 'Unknown')
                print(f"  - {layer_name:25} ({layer_type})")

    # Load with Keras for structural summary
    try:
        model = load_model(file_path, compile=False)
        print("\n--- Keras Model Summary ---")
        model.summary(line_length=150)
    except Exception as e:
        print("\n⚠️ Could not load full model for summary:", e)


# Example: replace with your actual file path
inspect_h5_model("/kaggle/working/final_coevo_iot_ids_model.h5")

```


Historical extension (inactive; see backups).

```python
# ===========================================================
# 🎨 VISUALIZATION: FEDERATED LEARNING IN CO-EVOL IDS
# ===========================================================
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def visualize_federated_learning_architecture():
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.axis("off")

    # Global Aggregator (Server)
    server = patches.Rectangle((4, 7.5), 3, 1, fc="#6ea8fe", ec="black", lw=2)
    ax.add_patch(server)
    ax.text(5.5, 8.0, "🌐 Global Server\n(Federated Aggregator)", ha="center", va="center", fontsize=11, fontweight='bold')

    # Co-Evolution Zone (Attacker–Defender Adaptation)
    evo = patches.Rectangle((3.5, 6), 4, 1, fc="#b6d7a8", ec="black", lw=2)
    ax.add_patch(evo)
    ax.text(5.5, 6.5, "♻️ Co-Evolution Zone\n(Attacker–Defender)", ha="center", va="center", fontsize=10, fontweight='bold')

    # Clients
    positions = [(1, 3), (3, 3), (5, 3), (7, 3), (9, 3)]
    for i, (x, y) in enumerate(positions):
        client_box = patches.Rectangle((x, y), 1.5, 1, fc="#ffd966", ec="black", lw=2)
        ax.add_patch(client_box)
        ax.text(x+0.75, y+0.5, f"Client-{i+1}\nIoT Node", ha="center", va="center", fontsize=9)

        # Arrow from client to server
        ax.arrow(x+0.75, y+1, 3.75-(x+0.75), 3.5, head_width=0.15, head_length=0.3, fc="gray", ec="gray")
        # Arrow back from server to client
        ax.arrow(5.5, 7.5, x+0.75-5.5, -3.5, head_width=0.15, head_length=0.3, fc="gray", ec="gray")

    plt.title("Federated Learning Integration with Co-Evolutionary CNN–BiLSTM IDS", fontsize=13, fontweight='bold')
    plt.show()

visualize_federated_learning_architecture()

```


Historical extension (inactive; see backups).

```python
# ===========================
# Co-Evolutionary Federated Learning IDS Network Architecture
# ===========================
from graphviz import Digraph

dot = Digraph("CoEvo_Federated_IDS", format="png")
dot.attr(rankdir='TB', bgcolor='white')

# ----------------------------
# Layer 1: IoT Network Layer
# ----------------------------
iot_nodes = ["IoT Device 1", "IoT Device 2", "IoT Device 3", "IoT Device 4", "IoT Device 5"]
for i, dev in enumerate(iot_nodes, 1):
    dot.node(f"I{i}", dev, shape="ellipse", style="filled", color="#06d6a0", fontcolor="white")

# ----------------------------
# Layer 2: Edge/Federated Clients
# ----------------------------
clients = [f"Client {i}\n(Local Edge Node)" for i in range(1, 6)]
for i, client in enumerate(clients, 1):
    dot.node(f"C{i}", client, shape="box", style="filled,rounded", color="#118ab2", fontcolor="white", width="1.8")

# ----------------------------
# Layer 3: Local Training Units
# ----------------------------
for i in range(1, 6):
    dot.node(f"L{i}", "Local Training\n(CNN–BiLSTM + Game Adaptation)", shape="rect", style="rounded", color="#06b6d4", fontcolor="black")

# ----------------------------
# Layer 4: Federated Aggregator
# ----------------------------
dot.node("FA", "Federated Aggregation Server\n(Weighted Model Fusion)", shape="oval", style="filled", color="#073b4c", fontcolor="white", width="2.4")

# ----------------------------
# Layer 5: Global Model & Adaptation
# ----------------------------
dot.node("GM", "Global Model\n(Co-Evolutionary CNN–BiLSTM)", shape="box", style="filled", color="#023047", fontcolor="white", width="2.4")
dot.node("AD", "Co-Evolutionary Adaptation\n(Attacker ↔ Defender)", shape="diamond", style="filled", color="#ef476f", fontcolor="white")

# ----------------------------
# Layer 6: Decision & Output
# ----------------------------
dot.node("OUT", "Adaptive Global IDS\n(Real-time Intrusion Detection)", shape="box", style="filled,rounded", color="#118ab2", fontcolor="white")

# ----------------------------
# Connections
# ----------------------------

# IoT Devices -> Clients
for i in range(1, 6):
    dot.edge(f"I{i}", f"C{i}", label="Raw IoT Traffic", color="#06d6a0")

# Clients -> Local Training
for i in range(1, 6):
    dot.edge(f"C{i}", f"L{i}", label="Feature Extraction & Preprocessing", color="#118ab2")

# Local Training -> Federated Aggregator
for i in range(1, 6):
    dot.edge(f"L{i}", "FA", label="Model Update", color="#219ebc")

# Federated Aggregator -> Global Model
dot.edge("FA", "GM", label="Aggregated Global Model", color="#023047")

# Global Model ↔ Adaptation Loop
dot.edge("GM", "AD", label="Security Feedback Loop", color="#ef476f")
dot.edge("AD", "GM", label="Adaptive Strategy Update", color="#ef476f")

# Global Model -> Output
dot.edge("GM", "OUT", label="Final Predictions", color="#118ab2")

# ----------------------------
# Render Diagram
# ----------------------------
dot.render("/kaggle/working/CoEvo_Federated_IDS_Network", view=True)
print("✅ Clean Network Architecture Diagram Generated — saved as CoEvo_Federated_IDS_Network.png")

```
